# Dự đoán giá nhà

# 1. Problem: dự đoán giá nhà

## Tiền xử lý data

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/homedata.csv")
print(df.head())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [ ]:
# Feature Engineering
features = ['OverallQual', 'GrLivArea','TotalBsmtSF', 'GarageCars', 'GarageArea', 'YearBuilt','YearRemodAdd','FullBath','TotRmsAbvGrd','Fireplaces']
df_features = df[features + ['SalePrice']].copy()
# Remove outliers using IQR method across selected numeric features
Q1 = df_features.quantile(0.25)
Q3 = df_features.quantile(0.75)
IQR = Q3 - Q1
mask = ~((df_features < (Q1 - 1.5 * IQR)) | (df_features > (Q3 + 1.5 * IQR))).any(axis=1)
df_no_outlier = df_features[mask].reset_index(drop=True)
print('Original shape:', df_features.shape)
print('No-outlier shape:', df_no_outlier.shape)
X_outliner = df[features]
Y_outliner = df['SalePrice']
X = df_no_outlier[features]
y = df_no_outlier['SalePrice']
print(X.head())
X.info()
print(y.head())
y.info()

Original shape: (1460, 11)
No-outlier shape: (1308, 11)
   OverallQual  GrLivArea  TotalBsmtSF  GarageCars  GarageArea  YearBuilt  \
0            7       1710          856           2         548       2003   
1            6       1262         1262           2         460       1976   
2            7       1786          920           2         608       2001   
3            7       1717          756           3         642       1915   
4            8       2198         1145           3         836       2000   

   YearRemodAdd  FullBath  TotRmsAbvGrd  Fireplaces  
0          2003         2             8           0  
1          1976         2             6           1  
2          2002         2             6           1  
3          1970         1             7           1  
4          2000         2             9           1  
<class 'pandas.DataFrame'>
RangeIndex: 1308 entries, 0 to 1307
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        -

In [11]:
# Normalize data
from sklearn.preprocessing import Normalizer
sc = Normalizer()

# No outliner
x_scale = sc.fit_transform(X_outliner)
x_no_outlier_scaled = sc.fit_transform(X)

print(x_scale)

[[2.02230404e-03 4.94019987e-01 2.47298894e-01 ... 5.77801154e-04
  2.31120462e-03 0.00000000e+00]
 [1.79235320e-03 3.76991624e-01 3.76991624e-01 ... 5.97451068e-04
  1.79235320e-03 2.98725534e-04]
 [1.98642846e-03 5.06823033e-01 2.61073455e-01 ... 5.67550989e-04
  1.70265297e-03 2.83775495e-04]
 ...
 [1.82836415e-03 6.11196014e-01 3.00896499e-01 ... 5.22389756e-04
  2.35075390e-03 5.22389756e-04]
 [1.56799640e-03 3.38060023e-01 3.38060023e-01 ... 3.13599279e-04
  1.56799640e-03 0.00000000e+00]
 [1.51073529e-03 3.79496705e-01 3.79496705e-01 ... 3.02147058e-04
  1.81288235e-03 0.00000000e+00]]


In [13]:
# Split du lieu
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_scale, Y_outliner, test_size=0.3, random_state=42)
X_No_train, X_No_test, y_NO_train, y_NO_test = train_test_split(x_no_outlier_scaled, y, test_size=0.3, random_state=42)

In [ ]:
# Training với Linear Regression, Decision tree, Random Forest, Gradient Boosting
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=50),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=50),
}


for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"({name}) with outliers - R^2 Score: ({r2}), MAE: ({mae})")
    model.fit(X_No_train, y_NO_train)
    y_no_outliers_pred = model.predict(X_No_test)
    r2_no_outliers = r2_score(y_NO_test, y_no_outliers_pred)
    mae_no_outliers = mean_absolute_error(y_NO_test, y_no_outliers_pred)
    print(
        f" ({name}) without outliers - R^2 Score: ({r2_no_outliers}), MAE: ({mae_no_outliers})"
    )


(Linear Regression) with outliers - R^2 Score: (0.7815896170956301), MAE: (25875.469763652392)
 (Linear Regression) without outliers - R^2 Score: (0.7932762876176951), MAE: (20289.43284343791)
(Decision Tree) with outliers - R^2 Score: (0.7516269559735617), MAE: (27178.76484018265)
 (Decision Tree) without outliers - R^2 Score: (0.6460643523868406), MAE: (24444.43765903308)
(Random Forest) with outliers - R^2 Score: (0.8605447406135901), MAE: (20173.93434659709)
 (Random Forest) without outliers - R^2 Score: (0.8129879348561655), MAE: (18233.325994789775)
(Gradient Boosting) with outliers - R^2 Score: (0.8690081134191934), MAE: (20697.918923232482)
 (Gradient Boosting) without outliers - R^2 Score: (0.8230085260160346), MAE: (18363.498390676046)


Decision Tree - Mean Squared Error: 1359089861.0821917, R^2 Score: 0.8052347951112586
Random Forest - Mean Squared Error: 769074179.8360906, R^2 Score: 0.8897873536550809
Gradient Boosting - Mean Squared Error: 768714583.1178881, R^2 Score: 0.8898388858830104
